# YOLOv8 Skin Disease Detection — Object Detection Approach

In this notebook, I train a **YOLOv8 Medium** object detection model on a custom skin disease dataset to detect and localize skin conditions in images. Unlike my EfficientNet-B0 classifier (which classifies the entire image), this YOLO approach draws bounding boxes around affected areas — making it far more useful for real-world clinical analysis.

## Why YOLOv8?

YOLO (You Only Look Once) is a single-pass detection architecture that processes the entire image in one forward pass, making it extremely fast while maintaining high accuracy. I chose **YOLOv8m** (medium variant) for the best balance between speed and detection performance.

## What This Notebook Covers

- GPU setup and environment configuration
- Installing Ultralytics YOLOv8
- Preparing the skin disease dataset
- Training YOLOv8m for 80 epochs on 11 skin condition classes
- Evaluating model performance (confusion matrix, PR curves, mAP)
- Running inference on test images

**Let's get started.**

## GPU Check

First, I verify that the GPU is available. I used dual Tesla T4 GPUs on Kaggle for this training run. If you're on Colab, go to `Edit` → `Notebook settings` → `Hardware accelerator` → `GPU`.

In [ ]:
!nvidia-smi

Sun Sep  1 17:19:14 2024       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.07              Driver Version: 550.90.07      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8              9W /   70W |       1MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Setting the working directory
import os
HOME = os.getcwd()
print(HOME)

## Install YOLOv8

Installing the `ultralytics` package which provides the YOLOv8 implementation. This handles everything — model architecture, training loops, data loading, augmentation, and inference.

In [ ]:
# Install ultralytics and verify setup
!pip install ultralytics

from IPython import display
display.clear_output()

import ultralytics
ultralytics.checks()

In [ ]:
# Import YOLO model class and display utilities
from ultralytics import YOLO
from IPython.display import display, Image

## YOLOv8 CLI Overview

YOLOv8 provides a clean CLI for training, validation, prediction, and export — no boilerplate code needed.

The YOLO CLI supports multiple tasks and modes:

```
yolo task=detect    mode=train    model=yolov8n.yaml      args...
          classify       predict        yolov8n-cls.yaml  args...
          segment        val            yolov8n-seg.yaml  args...
                         export         yolov8n.pt        format=onnx  args...
```

I'm using `task=detect` since I want bounding box detection around skin lesions, not just classification.

## Pre-trained COCO Baseline

Before training on skin data, YOLOv8 comes pre-trained on COCO (80 common object classes). I use this as the backbone and fine-tune it on my skin disease dataset via transfer learning.

## Preparing the Skin Disease Dataset

I used a custom skin disease dataset with 11 classes of skin conditions, annotated with bounding boxes in YOLO format. The dataset is structured with `train/`, `valid/`, and `test/` splits, each containing `images/` and `labels/` directories. The `data.yaml` file defines class names and paths.

In [ ]:
# Dataset should be in the working directory with YOLO-format annotations
# Structure: skin-diseases-1/{train,valid,test}/{images,labels}/
# If using a zipped dataset, uncomment below:
# !unzip -q skin-diseases-1.zip -d .

## Training YOLOv8m on Skin Disease Data

Training the **YOLOv8 Medium** model for **80 epochs** at **800x800** image resolution. I went with the medium variant (`yolov8m`) because it has 25.8M parameters — enough capacity to learn the visual patterns of different skin conditions while still being fast enough for real-time inference.

**Training config:**
- **Model:** YOLOv8m (25.8M params, 78.7 GFLOPs)
- **Epochs:** 80
- **Image Size:** 800x800
- **Dataset:** 11 skin disease classes with bounding box annotations

In [ ]:
%cd {HOME}

# Train YOLOv8m on skin disease dataset — 80 epochs, 800px images
!yolo task=detect mode=train model=yolov8m.yaml data=/kaggle/input/yaml-file/data.yaml epochs=80 imgsz=800 plots=True

In [ ]:
# Check all training artifacts generated
!ls {HOME}/runs/detect/train/

In [ ]:
%cd {HOME}
# Confusion matrix — shows per-class detection accuracy and misclassifications
Image(filename=f'{HOME}/runs/detect/train/confusion_matrix.png', width=600)

In [ ]:
%cd {HOME}
# Training curves — loss, precision, recall, mAP over 80 epochs
Image(filename=f'{HOME}/runs/detect/train/results.png', width=600)

In [ ]:
%cd {HOME}
# Validation predictions — bounding boxes drawn on actual skin disease images
Image(filename=f'{HOME}/runs/detect/train/val_batch0_pred.jpg', width=600)

## Validation Results

Running validation on the held-out validation set to get per-class metrics. Key metrics I'm looking at:
- **mAP50** — mean Average Precision at IoU 0.5 (how well it detects lesions)
- **Precision & Recall** — per-class detection reliability
- **Inference speed** — how fast can it process each image

In [ ]:
%cd {HOME}

# Validate the best model weights on the validation set
!yolo task=detect mode=val model={HOME}/runs/detect/train/weights/best.pt data=/kaggle/input/yaml-file/data.yaml

## Inference on Test Images

Now I run the trained model on unseen test images to see how it performs in practice. The model draws bounding boxes around detected skin conditions with confidence scores.

In [ ]:
%cd {HOME}
# Run inference on test images with 0.25 confidence threshold
!yolo task=detect mode=predict model={HOME}/runs/detect/train/weights/best.pt conf=0.25 source=/kaggle/working/skin-diseases-1/test/images save=True

## Deployment Notes

The trained model weights are saved at `runs/detect/train/weights/best.pt`. This `.pt` file can be:

- Loaded directly with `ultralytics` for Python inference
- Exported to ONNX/TensorRT for production deployment
- Integrated into the main SkinCare AI Flask app as an additional detection pipeline

The key advantage of this YOLO approach over my EfficientNet classifier is **localization** — YOLO doesn't just tell you *what* the condition is, it shows you *where* on the skin it's detected with bounding boxes and confidence scores. This makes it much more useful for clinical assessment where pinpointing affected areas matters.

In [ ]:
# To export the model for production:
# !yolo export model=runs/detect/train/weights/best.pt format=onnx